# Multi-Modal Product Review Analyzer - Demo

This notebook demonstrates how to use the Multi-Modal Product Review Analyzer to predict product ratings from images and text reviews.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import BertTokenizer, ViTImageProcessor

from model import create_model
from data_loader import prepare_sample_data, create_data_loaders
from train import Trainer
from evaluate import Evaluator, Predictor, load_model

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Create Sample Dataset

In [ ]:
# Create sample data for demonstration
df = prepare_sample_data('../data', num_samples=200)
print(f"Created {len(df)} sample reviews")
print("\nDataset preview:")
display(df.head())

# Rating distribution
print("\nRating distribution:")
print(df['rating'].value_counts().sort_index())

## 3. Create Model

In [ ]:
# Model configuration
config = {
    'image_model': 'vit',
    'text_model': 'distilbert',
    'fusion_dim': 512,
    'num_classes': 5,
    'dropout': 0.3,
    'freeze_backbones': True
}

# Create model
model = create_model(config)
print(f"Model created!")
print(f"Model size: {model.get_model_size():.2f} MB")

## 4. Prepare Data Loaders

In [ ]:
# Split data
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

# Save splits
train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    '../data/train.csv',
    '../data/val.csv',
    '../data/test.csv',
    '../data/images',
    batch_size=8
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 5. Visualize Sample Data

In [ ]:
# Get a batch
batch = next(iter(train_loader))

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(min(8, len(batch['image']))):
    # Denormalize image (for ViT)
    img = batch['image'][i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    
    axes[i].imshow(img)
    axes[i].set_title(f"Rating: {batch['label'][i].item() + 1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 6. Train Model (Small Demo)

Note: For full training, use the `train.py` script.

In [ ]:
# Training configuration
train_config = {
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'batch_size': 8,
    'use_amp': False  # Disable for demo
}
train_config.update(config)

# Create trainer
trainer = Trainer(model, train_loader, val_loader, train_config)

# Train for a few epochs (demo)
history = trainer.train(num_epochs=2, save_dir='../models')

print("Training completed!")

## 7. Visualize Training History

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluate Model

In [ ]:
# Create evaluator
evaluator = Evaluator(model, test_loader)

# Evaluate
results = evaluator.evaluate()

# Print report
evaluator.print_report(results)

# Plot confusion matrix
evaluator.plot_confusion_matrix(results['confusion_matrix'])
plt.show()

## 9. Make Predictions on New Samples

In [ ]:
# Create predictor
predictor = Predictor(model)

# Get a test sample
test_batch = next(iter(test_loader))

# Make predictions
for i in range(min(5, len(test_batch['image']))):
    image = test_batch['image'][i]
    input_ids = test_batch['input_ids'][i]
    attention_mask = test_batch['attention_mask'][i]
    true_label = test_batch['label'][i].item() + 1
    
    # Predict
    pred_rating, confidence = predictor.predict(image, input_ids, attention_mask)
    
    print(f"\nSample {i+1}:")
    print(f"  True Rating: {true_label}")
    print(f"  Predicted Rating: {pred_rating}")
    print(f"  Confidence: {confidence}")
    print(f"  Match: {'✓' if pred_rating == true_label else '✗'}")

## 10. Model Analysis

In [ ]:
# Analyze per-class performance
report = results['classification_report']

ratings = []
f1_scores = []

for i in range(1, 6):
    class_name = f'Rating {i}'
    if class_name in report:
        ratings.append(i)
        f1_scores.append(report[class_name]['f1-score'])

# Plot per-class F1 scores
plt.figure(figsize=(10, 6))
plt.bar(ratings, f1_scores, color='skyblue', edgecolor='navy')
plt.xlabel('Rating Class')
plt.ylabel('F1 Score')
plt.title('Per-Class F1 Scores')
plt.xticks(ratings)
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.show()

## Summary

This notebook demonstrated:
1. Creating a multi-modal dataset with images and text
2. Building a fusion model with pre-trained backbones
3. Training the model on product reviews
4. Evaluating performance with various metrics
5. Making predictions on new samples

For production use:
- Use a larger dataset (20K+ samples)
- Train for more epochs (10-20)
- Use GPU acceleration
- Experiment with different fusion strategies
- Fine-tune hyperparameters